# Difference equations

A difference equation defines each term from earlier ones. The characteristic roots decide whether solutions decay, oscillate, or explode, which is exactly the stability question for AR models.

Companion note: [difference_equations.md](../../notes/time_series/difference_equations.md).

In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
plt.rcParams['figure.figsize'] = (8, 3.5)

# First-order: x_t = 0.7 x_{t-1} + 2, x_0 = 10
a, c, x0 = 0.7, 2.0, 10.0
xstar = c / (1 - a)
x = [x0]
for t in range(1, 21):
    x.append(a * x[-1] + c)
x = np.array(x)
closed = xstar + a ** np.arange(21) * (x0 - xstar)
print('x1..x3 =', x[1:4], ' equilibrium =', round(xstar, 4))
print('recursion matches closed form:', np.allclose(x, closed))
plt.plot(x, 'o-'); plt.axhline(xstar, color='r', ls='--'); plt.title('stable first-order recursion'); plt.show()

x1..x3 = [9.   8.3  7.81]  equilibrium = 6.6667
recursion matches closed form: True


/tmp/ipykernel_78575/43407346.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.plot(x, 'o-'); plt.axhline(xstar, color='r', ls='--'); plt.title('stable first-order recursion'); plt.show()


The deviation from equilibrium shrinks by the factor $0.7$ each step. Now a second-order equation, $a_n=5a_{n-1}-6a_{n-2}$ with $a_0=4,a_1=10$: the characteristic polynomial is $\lambda^2-5\lambda+6$.

In [2]:
roots = np.roots([1, -5, 6])
print('roots:', roots)
A = np.array([[1, 1], [roots[0], roots[1]]])
c1, c2 = np.linalg.solve(A, [4, 10])
n_ = np.arange(10)
closed = c1 * roots[0] ** n_ + c2 * roots[1] ** n_
rec = [4, 10]
for k in range(2, 10):
    rec.append(5 * rec[-1] - 6 * rec[-2])
print('c1, c2 =', c1, c2)
print('recursion  :', rec)
print('closed form:', np.round(closed).astype(int))

roots: [3. 2.]
c1, c2 = 1.9999999999999998 2.0000000000000004
recursion  : [4, 10, 26, 70, 194, 550, 1586, 4630, 13634, 40390]
closed form: [    4    10    26    70   194   550  1586  4630 13634 40390]


Both roots have modulus above one, so the sequence explodes ($2\cdot2^n+2\cdot3^n$). Compare the three qualitative regimes for a general AR(2)-type recursion via the root modulus.

In [3]:
def path(phi1, phi2, n=40, x0=(1.0, 1.0)):
    x = list(x0)
    for _ in range(n):
        x.append(phi1 * x[-1] + phi2 * x[-2])
    return np.array(x)

cases = {'real roots, stable (0.5, 0.3)': (0.5, 0.3),
         'complex roots, stable (1.0, -0.5)': (1.0, -0.5),
         'complex roots, unstable (1.0, -1.2)': (1.0, -1.2)}
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (name, (p1, p2)) in zip(axes, cases.items()):
    r = np.roots([1, -p1, -p2])
    ax.plot(path(p1, p2), 'o-', ms=3)
    ax.set_title(name + f'\n|roots|={np.round(np.abs(r), 2)}', fontsize=9)
plt.tight_layout(); plt.show()

/tmp/ipykernel_78575/2253736140.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Stability requires every characteristic root to lie inside the unit circle; complex roots add oscillation. Finally, the Fibonacci-type recurrence $a_n=a_{n-1}+a_{n-2}$, $a_0=2,a_1=3$ against the Binet-style closed form in the note.

In [4]:
s5 = np.sqrt(5)
phi_, psi_ = (1 + s5) / 2, (1 - s5) / 2
c1, c2 = 1 - 2 / s5, 1 + 2 / s5          # coefficients for psi_^n and phi_^n
n_ = np.arange(10)
closed = c1 * psi_ ** n_ + c2 * phi_ ** n_
shifted = (phi_ ** (n_ + 3) - psi_ ** (n_ + 3)) / s5
print(np.round(closed).astype(int))
print(np.allclose(closed, shifted))

[  2   3   5   8  13  21  34  55  89 144]
True
